# ЛР6 · AI-детектив «Ладога»

Лаптев Егор Игоревич (408981), Поляков Александр Владимирович (409367), К3439.

**LLM-прогон в этом файле ещё не выполнен.** Код и исходные данные встроены. Выберите GPU в Colab и выполните все ячейки. Доступ к почте/Drive не нужен. Параметры одинаковы для A/B/C, ответы сохраняются по одному. Итоговые stress-метрики ждут проверки двумя участниками.

Перед исследовательским прогоном оба участника должны проверить gold по docs/review_checklist.md. Проверка кодом не подтверждает семантическую корректность эталонов.

## 1. Восстановить проект

Данные полностью вымышлены. Gold и рабочая истина не передаются модели.

In [ ]:
#@title Восстановить встроенный проект
import base64,io,zipfile,os
from pathlib import Path
BASE=Path('/content') if Path('/content').exists() else Path.cwd()
PAYLOAD='...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    archive.extractall(BASE)
PROJECT=BASE/'ai-detective'
os.chdir(PROJECT)
print('Проект:', PROJECT)

## 2. Зависимости

Эта ячейка скачивает библиотеки. Модель загружается отдельно в разделе 4.

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-model.txt'],check=True)

## 3. Проверить корпус и реализацию

Эти проверки не являются измерением качества LLM.

In [ ]:
subprocess.run([sys.executable,'validate_data.py'],check=True)
subprocess.run([sys.executable,'-m','pytest','-q'],check=True)
import pandas as pd
display(pd.read_csv('docs/retrieval_audit.csv')[['question_id','gold_coverage','missing_ids']])

## 4. Загрузить одну модель

Qwen2.5-7B-Instruct, NF4, greedy decoding, максимум 384 новых токена. KV cache выгружается в RAM. Полный контекст не обрезается. Объём GPU и время полного запуска ещё не измерены. Если GPU нет, выполнение остановится с явной причиной.

In [ ]:
import torch
from transformers import set_seed
set_seed(42)
assert torch.cuda.is_available(), 'Выберите GPU: Среда выполнения → Сменить среду выполнения'
print(torch.cuda.get_device_name(0))
from src.llm import TransformersLLM
llm=TransformersLLM('Qwen/Qwen2.5-7B-Instruct')
print('Ревизия модели:',llm.revision)

## 5. Все три режима и три вида изменений

120 запросов: 36 базовых, 36 перестановок, 12 удалений, 36 добавлений. Непроверенные варианты сохраняются как сырые результаты и не входят в итоговую stress-статистику. Повторный запуск продолжит по сохранённым записям. Не отключайте runtime, пока не скачали результаты.

In [ ]:
from src.experiment import run
OUTPUT=PROJECT/'results/qwen25_7b_nf4_v1'
rows=run(llm,OUTPUT)
print('Сохранено запросов:',len(rows))

## 6. Фактические метрики и черновик отчёта

Черновик строится только после вызовов модели. Его нужно дополнить разбором ошибок и совместной проверкой gold.

In [ ]:
from src.evaluation import summarize
from make_report import make_report
_,summary=summarize(rows)
display(summary)
make_report(rows,OUTPUT)
print('Непроверенные stress-варианты: provisional_includes_unreviewed.csv')

## 7. Скачать результаты

Архив содержит исходные ответы, промпты, снимки вариантов и черновик отчёта. Автоматическое скачивание не запускается, чтобы не прерывать Run all. Скачайте ZIP из панели файлов Colab. До отключения runtime сохраните архив.

In [ ]:
import shutil
archive=shutil.make_archive(str(BASE/'Lab6_results'),'zip',PROJECT)
print('Готовый архив:',archive)
from IPython.display import FileLink,display
display(FileLink(archive))

## После запуска

Оба участника проверяют docs/review_checklist.md и создают review.json только по факту проверки. Пересчёт: `python analyze_results.py --results results/qwen25_7b_nf4_v1 --review review.json`. Сохранённые raw-ответы не меняются. `report_draft.pdf` остаётся черновиком до содержательного анализа.